<a href="https://colab.research.google.com/github/mirian2004/AI-AI-/blob/Day13/Day13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**문서 세트 준비와 Chunking**

검색 단위를 만들기 위해 문서를 조각(Chunk)으로 구분

In [1]:
!pip install langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. 예시용 긴 문장 생성 (가상의 'Gemini 사용 설명서')
long_text = """
1. Gemini 모델 개요
Gemini는 구글에서 개발한 차세대 멀티모달 AI 모델입니다. 텍스트뿐만 아니라 이미지, 오디오, 비디오, 코드를 이해하고 처리할 수 있는 능력을 갖추고 있습니다.
가장 큰 특징은 방대한 컨텍스트 창을 지원하여 수천 페이지의 문서를 한 번에 이해할 수 있다는 점입니다.

2. RAG(검색 증강 생성)의 중요성
AI 모델은 학습된 시점 이후의 최신 정보를 알지 못하는 '지식 컷오프' 현상이 발생합니다.
이를 해결하기 위해 RAG 기술을 사용합니다. RAG는 모델 외부에서 벡터 데이터베이스를 구축하고, 질문과 관련된 문서를 실시간으로 검색하여 AI에게 전달합니다.
이 과정은 크게 로드(Load), 분할(Split), 임베딩(Embed), 저장(Store), 검색(Retrieve), 생성(Generate) 단계로 나뉩니다.
"""

# 2. 문서 분할
# 긴 문장을 200자 단위로 쪼개고, 문맥 유지를 위해 20자씩 겹치게 설정합니다.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
docs = text_splitter.create_documents([long_text])
print(f"문서 조각 개수: {len(docs)}개")


문서 조각 개수: 3개


**임베딩 생성**

In [6]:
import os
import numpy as np
import google.generativeai as genai
from google.colab import userdata

# API 키는 환경변수로 관리 권장
genai.configure(api_key=userdata.get('gemini_api_key'))

def gemini_embed_texts(texts, model="models/gemini-embedding-001"):
  vectors = []
  for t in texts:
    res = genai.embed_content(model = model, content = t)
    vectors.append(res["embedding"])
  return np.array(vectors, dtype="float32")

doc_texts = [d.page_content for d in docs]
doc_vecs = gemini_embed_texts(doc_texts)

print("임베딩 shape:", doc_vecs.shape)

임베딩 shape: (3, 3072)


**FAISS로 유사 문서 조각 찾기**

질문 벡터와 가장 가까운 벡터를 빠르게 탐색

In [11]:
!pip install faiss-cpu
import faiss

# FAISS 인덱스 생성
dim = doc_vecs.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(doc_vecs)

def retrieve(query, k=2):
  q_vec = gemini_embed_texts([query])[0].reshape(1,-1)
  distances, idxs = index.search(q_vec, k)
  retrieved_chunks = [doc_texts[i] for i in idxs[0]]
  return retrieved_chunks

**Context를 넣고 RAG 답변 출력**

사용자 질문에 대한 근거 기반 답변을 추출

In [15]:
def ask_question(query, k=2):
    retrieved_chunks = retrieve(query, k=k)
    context = "\n".join(retrieved_chunks)

    model = genai.GenerativeModel('gemini-flash-latest')
    prompt = f"""아래의 참고 정보를 바탕으로 질문에 답하세요. 정보에 없는 내용은 모른다고 답하세요.

  [참고정보]
  {context}

  질문 : {query}
  """

    response = model.generate_content(prompt)
    return response.text, context

In [16]:
# [시나리오 1]
question = "RAG의 과정을 알려줘"
answer, retrieved_context = ask_question(question, k=2)
print("[AI 답변]")
print(answer)
print("\n[답변 근거]")
print(retrieved_context)

[AI 답변]
참고 정보에 따른 RAG의 과정은 크게 다음과 같습니다.

1. **로드 (Load)**
2. **분할 (Split)**
3. **임베딩 (Embed)**
4. **저장 (Store)**
5. **검색 (Retrieve)**
6. **생성 (Generate)**

[답변 근거]
2. RAG(검색 증강 생성)의 중요성
AI 모델은 학습된 시점 이후의 최신 정보를 알지 못하는 '지식 컷오프' 현상이 발생합니다.
이를 해결하기 위해 RAG 기술을 사용합니다. RAG는 모델 외부에서 벡터 데이터베이스를 구축하고, 질문과 관련된 문서를 실시간으로 검색하여 AI에게 전달합니다.
이 과정은 크게 로드(Load), 분할(Split), 임베딩(Embed), 저장(Store), 검색(Retrieve), 생성(Generate) 단계로 나뉩니다.


In [17]:
# [시나리오 2]
question = "chatGPT가 뭐야?"
answer, retrieved_context = ask_question(question, k=2)
print("[AI 답변]")
print(answer)
print("\n[답변 근거]")
print(retrieved_context)

[AI 답변]
제공된 참고 정보에 chatGPT에 대한 내용이 없으므로 모릅니다.

[답변 근거]
1. Gemini 모델 개요
Gemini는 구글에서 개발한 차세대 멀티모달 AI 모델입니다. 텍스트뿐만 아니라 이미지, 오디오, 비디오, 코드를 이해하고 처리할 수 있는 능력을 갖추고 있습니다.
가장 큰 특징은 방대한 컨텍스트 창을 지원하여 수천 페이지의 문서를 한 번에 이해할 수 있다는 점입니다.
2. RAG(검색 증강 생성)의 중요성
AI 모델은 학습된 시점 이후의 최신 정보를 알지 못하는 '지식 컷오프' 현상이 발생합니다.
이를 해결하기 위해 RAG 기술을 사용합니다. RAG는 모델 외부에서 벡터 데이터베이스를 구축하고, 질문과 관련된 문서를 실시간으로 검색하여 AI에게 전달합니다.
